# 01 — Almacenamiento en Google Cloud Storage

**Proyecto GCP:** `my-first-project-492901`  
**Bucket:** `big-data-proyecto-parcial`  
**Región:** `us-central1`

Este notebook crea el bucket en GCS y sube los 26 archivos CSV normalizados del dataset Chicago Crimes (2001–2026), organizados bajo el prefijo `raw/`.

```
gs://big-data-proyecto-parcial/
└── raw/
    ├── Chicago_Crimes_2001.csv
    ├── Chicago_Crimes_2002.csv
    ├── ...
    └── Chicago_Crimes_2026.csv
```

### Autenticación
Se usa **Application Default Credentials (ADC)** vía `gcloud auth application-default login` — no se necesita archivo JSON de credenciales.

In [1]:
# !pip install google-cloud-storage

In [2]:
import os
import time
from google.cloud import storage

PROJECT_ID  = 'my-first-project-492901'
BUCKET_NAME = 'big-data-proyecto-parcial'
REGION      = 'us-central1'
LOCAL_FOLDER = 'Chicago_Crimes_by_Year'
GCS_PREFIX   = 'raw'

# ADC: usa las credenciales de 'gcloud auth application-default login'
client = storage.Client(project=PROJECT_ID)
print(f'Cliente GCS inicializado — proyecto: {PROJECT_ID}')

C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Cliente GCS inicializado — proyecto: my-first-project-492901


In [3]:
# ── Crear bucket si no existe ────────────────────────────────────────────────
bucket = client.bucket(BUCKET_NAME)

if not bucket.exists():
    bucket = client.create_bucket(BUCKET_NAME, location=REGION)
    bucket.storage_class = 'STANDARD'
    bucket.patch()
    print(f'Bucket creado:    gs://{BUCKET_NAME}  [{REGION}]')
else:
    print(f'Bucket existente: gs://{BUCKET_NAME}')

print(f'Storage class:    {bucket.storage_class or "STANDARD"}')
print(f'Ubicación:        {bucket.location}')

Bucket creado:    gs://big-data-proyecto-parcial  [us-central1]
Storage class:    STANDARD
Ubicación:        US-CENTRAL1


In [4]:
# ── Subir todos los CSVs ─────────────────────────────────────────────────────
csv_files = sorted(f for f in os.listdir(LOCAL_FOLDER) if f.endswith('.csv'))
print(f'Archivos a subir: {len(csv_files)}')
print('─' * 72)

total_bytes = 0
t0 = time.time()

for filename in csv_files:
    local_path = os.path.join(LOCAL_FOLDER, filename)
    gcs_path   = f'{GCS_PREFIX}/{filename}'
    blob       = bucket.blob(gcs_path)
    size_mb    = os.path.getsize(local_path) / (1024 ** 2)
    total_bytes += os.path.getsize(local_path)

    blob.upload_from_filename(local_path, content_type='text/csv')
    print(f'  ✓  {filename:<40}  {size_mb:>7.1f} MB')

elapsed = time.time() - t0
print('─' * 72)
print(f'Total: {total_bytes/(1024**2):.1f} MB en {len(csv_files)} archivos  ({elapsed:.1f}s)')

Archivos a subir: 26
────────────────────────────────────────────────────────────────────────


  ✓  Chicago_Crimes_2001.csv                      94.4 MB


  ✓  Chicago_Crimes_2002.csv                     111.7 MB


  ✓  Chicago_Crimes_2003.csv                     111.0 MB


  ✓  Chicago_Crimes_2004.csv                     109.7 MB


...

  ✓  Chicago_Crimes_2022.csv                      56.2 MB


  ✓  Chicago_Crimes_2023.csv                      61.9 MB


  ✓  Chicago_Crimes_2024.csv                      60.4 MB


  ✓  Chicago_Crimes_2025.csv                      21.9 MB


  ✓  Chicago_Crimes_2026.csv                      12.5 MB
────────────────────────────────────────────────────────────────────────
Total: 1941.9 MB en 26 archivos  (149.0s)


In [5]:
# ── Verificación: listar objetos en el bucket ────────────────────────────────
print(f'Contenido de gs://{BUCKET_NAME}/{GCS_PREFIX}/')
print('─' * 65)

blobs = sorted(client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX + '/'), key=lambda b: b.name)
total_size = 0
for blob in blobs:
    size_mb = blob.size / (1024 ** 2)
    total_size += blob.size
    print(f'  {blob.name:<52}  {size_mb:>7.1f} MB')

print('─' * 65)
print(f'Total: {len(blobs)} objetos  |  {total_size/(1024**2):.1f} MB')
print()
print('CAPTURA 1 → Consola GCP > Cloud Storage > Buckets (nombre, región, clase)')
print('CAPTURA 2 → Dentro del bucket, carpeta raw/ con los 26 archivos')
print('CAPTURA 3 → Clic en un CSV individual (metadata: tamaño, MIME, URL)')
print('CAPTURA 4 → Pestaña Permissions del bucket')
print('CAPTURA 5 → Esta celda con el output de verificación')

Contenido de gs://big-data-proyecto-parcial/raw/
─────────────────────────────────────────────────────────────────


  raw/Chicago_Crimes_2001.csv                              94.4 MB
  raw/Chicago_Crimes_2002.csv                             111.7 MB
  raw/Chicago_Crimes_2003.csv                             111.0 MB
  raw/Chicago_Crimes_2004.csv                             109.7 MB
  raw/Chicago_Crimes_2005.csv                             106.1 MB
  raw/Chicago_Crimes_2006.csv                             104.9 MB
  raw/Chicago_Crimes_2007.csv                             102.3 MB
  raw/Chicago_Crimes_2008.csv                              99.5 MB
  raw/Chicago_Crimes_2009.csv                              91.7 MB
  raw/Chicago_Crimes_2010.csv                              86.9 MB
  raw/Chicago_Crimes_2011.csv                              82.5 MB
  raw/Chicago_Crimes_2012.csv                              78.8 MB
  raw/Chicago_Crimes_2013.csv                              72.2 MB
  raw/Chicago_Crimes_2014.csv                              64.9 MB
  raw/Chicago_Crimes_2015.csv                              62.

## Justificación del uso de Google Cloud Storage

| Criterio | Justificación |
|---|---|
| **Desacoplamiento cómputo/almacenamiento** | Dask, Spark y BigQuery leen desde GCS sin mover datos — arquitectura data lake estándar de la industria |
| **Durabilidad** | SLA de 99.999999999% (11 nueves) con replicación geográfica automática |
| **Escalabilidad** | El dataset actual (~3 GB) escala a TB sin cambiar arquitectura ni reprovisionar |
| **Integración nativa GCP** | BigQuery External Tables y Dataproc leen GCS sin ETL adicional |
| **Costo optimizado** | Datos históricos pre-2020 elegibles para clase Nearline (40% más barato) |
| **Partition pruning** | 26 archivos por año permiten leer solo los años necesarios en cada consulta |